In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from neo4j import GraphDatabase, basic_auth
import openai

driver = GraphDatabase.driver(
    "neo4j://52.4.166.125:7687",
    auth=basic_auth("neo4j", "stage-alkalinity-crashes")
)

In [4]:
cypher_query = '''
MATCH (m:Movie {title:$movie})<-[:RATED]-(u:User)-[:RATED]->(rec:Movie)
RETURN distinct rec.title AS recommendation LIMIT 20
'''

with driver.session(database='neo4j') as session:
    results = session.read_transaction(
        lambda tx: tx.run(cypher_query,
                          movie='Crimson Tide').data())
    for record in results:
        print(record['recommendation'])

<ipython-input-4-c1409c242bba>:7: DeprecationWarning: read_transaction has been renamed to execute_read
  results = session.read_transaction(


Mr. Holland's Opus
Apollo 13
Dead Man Walking
Seven (a.k.a. Se7en)
Heat
Get Shorty
Fugitive, The
Dave
Addams Family Values
True Lies
Speed
Lion King, The
Four Weddings and a Funeral
Forrest Gump
Star Trek: Generations
Shawshank Redemption, The
Stargate
Pulp Fiction
Outbreak
Miracle on 34th Street


In [5]:
from neo4j_genai.retrievers import Text2CypherRetriever
from neo4j_genai.llm import OpenAILLM

llm = OpenAILLM(model_name='gpt-4o', model_params={'temperature': 0})

In [6]:
from neo4j import GraphDatabase
from neo4j.time import Date

def get_node_datatype(value):
    '''
    입력된 노드 Value의 데이터 타입을 반환하는 함수
    '''
    if isinstance(value, str):
        return 'STRING'
    elif isinstance(value, int):
        return 'INTEGER'
    elif isinstance(value, float):
        return 'FLOAT'
    elif isinstance(value, bool):
        return 'BOOLEAN'
    elif isinstance(value, list):
        return f'LIST[{get_node_datatype(value[0])}]' if value else "LIST"
    elif isinstance(value, Date):
        return 'DATE'
    else:
        return 'UNKNOWN'

In [7]:
def get_schema(uri, user, password):
    '''
    Graph DB의 정보를 받아 노드 및 관계의 프로퍼티를 추출하고 스키마 딕셔너리를 반환하는 함수
    '''
    driver = GraphDatabase.driver(
        uri,
        auth=basic_auth(user, password)
    )

    with driver.session() as session:
        node_query = '''
        MATCH (n)
        WITH DISTINCT labels(n) AS node_labels, keys(n) AS property_keys, n
        UNWIND node_labels AS label
        UNWIND property_keys AS key
        RETURN label, key, n[key] AS sample_value
        '''
        nodes = session.run(node_query)

        rel_query = '''
        MATCH ()-[r]->()
        WITH DISTINCT type(r) AS rel_type, keys(r) AS property_keys, r
        UNWIND property_keys AS key
        RETURN rel_type, key, r[key] AS sample_value
        '''
        relationships = session.run(rel_query)
        
        rel_direction_query = '''
        MATCH (a)-[r]->(b)
        RETURN DISTINCT labels(a) AS start_label, type(r) AS rel_type, labels(b) AS end_label
        ORDER BY start_label, rel_type, end_label
        '''
        rel_directions = session.run(rel_direction_query)

        schema = {'nodes': {}, 'relationships': {}, 'relations': []}

        for record in nodes:
            label = record['label']
            key = record['key']
            sample_value = record['sample_value']
            inferred_type = get_node_datatype(sample_value)
            if label not in schema['nodes']:
                schema['nodes'][label] = {}
            schema['nodes'][label][key] = inferred_type
        
        for record in relationships:
            rel_type = record['rel_type']
            key = record['key']
            sample_value = record['sample_value']
            inferred_type = get_node_datatype(sample_value)
            if rel_type not in schema['relationships']:
                schema['relationships'][rel_type] = {}
            schema['relationships'][rel_type][key] = inferred_type
        
        for record in rel_directions:
            start_label = record['start_label'][0]
            rel_type = record['rel_type']
            end_label = record['end_label'][0]
            schema['relations'].append(f'(:{start_label})-[:{rel_type}]->(:{end_label})')
        
        return schema

def format_schema(schema):
    '''
        스키마 딕셔너리를 LLM에 제공하기 위해 원하는 형태로 formatting 하는 함수
    '''
    result = []

    result.append('Node properties:')
    for label, properties in schema['nodes'].items():
        props = ', '.join(f'{k}: {v}' for k, v in properties.items())
        result.append(f'{label} {{proprs}}')
    
    result.append('Relationship properties:')
    for rel_type, properties in schema['relationships'].items():
        props = ', '.join(f'{k}: {v}' for k, v in properties.items())
        result.append(f'{rel_type} {{{props}}}')
    
    result.append('The realtionships:')
    for relation in schema['relations']:
        result.append(relation)
    
    return '\n'.join(result)

In [8]:
schema = get_schema("neo4j://52.4.166.125:7687","neo4j", "stage-alkalinity-crashes")
neo4j_schema = format_schema(schema)
print(neo4j_schema)

Node properties:
Movie {proprs}
Genre {proprs}
User {proprs}
Actor {proprs}
Person {proprs}
Director {proprs}
Relationship properties:
RATED {rating: FLOAT, timestamp: INTEGER}
ACTED_IN {role: STRING}
DIRECTED {role: STRING}
The realtionships:
(:Actor)-[:ACTED_IN]->(:Movie)
(:Actor)-[:DIRECTED]->(:Movie)
(:Actor)-[:ACTED_IN]->(:Movie)
(:Director)-[:DIRECTED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)
(:User)-[:RATED]->(:Movie)


In [9]:
examples = [
    'USER INPUT: "Toy Story에 어떤 배우들이 출연하나요?" QUERY: MATCH (a:Actor)-[:ACTED_IN]->(m:Movie) WHERE m.title = "Toy Story" RETURN a.name',
    "USER INPUT: 'Toy Story의 평균 평점은 몇점인가요?' QUERY: MATCH (u:User)-[r:RATED]->(m:Movie) WHERE m.title = 'Toy Story' RETURN AVG(r.rating)",

    """
    USER INPUT: '저는 Toy Story 영화를 좋아합니다. Toy Story를 재밌게 본 사람은 또 어떤 영화를 재밌게 봤나요?'
    QUERY: MATCH (m:Movie)<-[r:RATED]-(u:User)-[recr:RATED]->(userBasedRec:Movie)
    WITH userBasedRec, COUNT(recr) AS recCount, AVG(recr.rating) AS avgRating
    ORDER BY avgRating DESC, recCount DESC
    RETURN DISTINCT userBasedRec.title, avgRating, recCount
    LIMIT 10
    """,

    """
    USER INPUT: '저는 'Wizard of Oz, The' 와 같은 영화를 좋아합니다. 이 영화와 비슷한 영화 추천해줄 수 있나요?'
    QUERY: MATCH (m:Movie) WHERE m.title = 'Wizard of Oz, The'
    MATCH (m)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie)
    WITH mm, rec, count(*) AS gs

    OPTIONAL MATCH (m)<-[:ACTED_IN]-(a)-[:ACTED_IN]->(rec)
    WITH m, rec, gs, count(a) AS as

    OPTIONAL MATCH (m)<-[:DIRECTED]-(d)-[:DIRECTED]->(rec)
    WITH m, rec, gs, as, count(d) AS ds

    RETURN rec.title AS recommendation,
            rec.poster AS rec_poster,
            gs AS genre_similarity,
            as AS actor_similarity,
            ds AS director_similarity,
            (5*gs)+(3*as)+(4*ds) AS score
    ORDER BY score DESC LIMIT 10
    """,

    """
    USER INPUT: '영화 Inception'과 비슷한 장르 혹은 비슷한 분위기의 영화를 추천해주세요.'
    QUERY: MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie)
    WHERE m.title = 'Inception' WITH rec, collect(g.name) AS genres, count(*) AS commonGenres
    RETURN rec.title, genres, commonGenres ORDER BY commonGenres DESC LIMIT 10;
    """
]

In [10]:
retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=neo4j_schema,
    examples=examples
)

query_text = 'Tom Hanks가 어떤 영화에 출연했나요?'
search_result = retriever.search(query_text=query_text)

In [11]:
search_result.items

[RetrieverResultItem(content="<Record m.title='Punchline'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Catch Me If You Can'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Dragnet'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Saving Mr. Banks'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Bachelor Party'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Volunteers'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Man with One Red Shoe, The'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Splash'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Big'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Nothing in Common'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Money Pit, The'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Toy Story of Terror'>", metadata=None),
 RetrieverResultItem(con

In [12]:
query_text = '저는 Titanic을 좋아합니다. 비슷한 영화를 추천해줄 수 있나요?'
search_result = retriever.search(query_text=query_text)

In [13]:
search_result.metadata['cypher']

"MATCH (m:Movie) WHERE m.title = 'Titanic'\nMATCH (m)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie)\nWITH m, rec, COUNT(g) AS genre_similarity\n\nOPTIONAL MATCH (m)<-[:ACTED_IN]-(a:Actor)-[:ACTED_IN]->(rec)\nWITH m, rec, genre_similarity, COUNT(a) AS actor_similarity\n\nOPTIONAL MATCH (m)<-[:DIRECTED]-(d:Director)-[:DIRECTED]->(rec)\nWITH rec, genre_similarity, actor_similarity, COUNT(d) AS director_similarity\n\nRETURN rec.title AS recommendation,\n       genre_similarity,\n       actor_similarity,\n       director_similarity,\n       (5*genre_similarity) + (3*actor_similarity) + (4*director_similarity) AS score\nORDER BY score DESC LIMIT 10"

In [14]:
print(search_result.metadata['cypher'])

MATCH (m:Movie) WHERE m.title = 'Titanic'
MATCH (m)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie)
WITH m, rec, COUNT(g) AS genre_similarity

OPTIONAL MATCH (m)<-[:ACTED_IN]-(a:Actor)-[:ACTED_IN]->(rec)
WITH m, rec, genre_similarity, COUNT(a) AS actor_similarity

OPTIONAL MATCH (m)<-[:DIRECTED]-(d:Director)-[:DIRECTED]->(rec)
WITH rec, genre_similarity, actor_similarity, COUNT(d) AS director_similarity

RETURN rec.title AS recommendation,
       genre_similarity,
       actor_similarity,
       director_similarity,
       (5*genre_similarity) + (3*actor_similarity) + (4*director_similarity) AS score
ORDER BY score DESC LIMIT 10


In [15]:
search_result.items

[RetrieverResultItem(content="<Record recommendation='Revolutionary Road' genre_similarity=2 actor_similarity=2 director_similarity=0 score=16>", metadata=None),
 RetrieverResultItem(content="<Record recommendation='Sense and Sensibility' genre_similarity=2 actor_similarity=1 director_similarity=0 score=13>", metadata=None),
 RetrieverResultItem(content="<Record recommendation='Reader, The' genre_similarity=2 actor_similarity=1 director_similarity=0 score=13>", metadata=None),
 RetrieverResultItem(content="<Record recommendation='Quills' genre_similarity=2 actor_similarity=1 director_similarity=0 score=13>", metadata=None),
 RetrieverResultItem(content='<Record recommendation="William Shakespeare\'s Romeo + Juliet" genre_similarity=2 actor_similarity=1 director_similarity=0 score=13>', metadata=None),
 RetrieverResultItem(content="<Record recommendation='Total Eclipse' genre_similarity=2 actor_similarity=1 director_similarity=0 score=13>", metadata=None),
 RetrieverResultItem(content="

In [17]:
from neo4j_genai.generation import GraphRAG

rag = GraphRAG(retriever=retriever, llm=llm)

In [18]:
query_text = 'Titanic과 비슷한 장르의 영화 추천해주세용.'

response = rag.search(query_text=query_text, return_context=True)
print('==== [Text2Cypher 를 통해 자동생성한 Cypher] ====')
print(response.retriever_result.metadata['cypher'])
print('\n==== [생성된 Cypher를 기반으로 최종답변생성] ====')
print(response.answer)

==== [Text2Cypher 를 통해 자동생성한 Cypher] ====
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie) WHERE m.title = 'Titanic' RETURN rec.title, COUNT(g) AS commonGenres ORDER BY commonGenres DESC LIMIT 10;

==== [생성된 Cypher를 기반으로 최종답변생성] ====
Here are some movies with genres similar to "Titanic":

1. "Hamlet" (14 common genres)
2. "Jane Eyre" (11 common genres)
3. "Carrie" (9 common genres)
4. "Misérables, Les" (8 common genres)
5. "Star Is Born, A" (7 common genres)
6. "Trip, The" (7 common genres)
7. "Phantom of the Opera, The" (7 common genres)
8. "Getaway, The" (7 common genres)
9. "3:10 to Yuma" (6 common genres)
10. "Emma" (6 common genres)
